In [6]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\lenovo\Downloads\DataAnalyst\DataAnalyst.csv")

In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253 entries, 0 to 2252
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         2253 non-null   int64  
 1   Job Title          2253 non-null   object 
 2   Salary Estimate    2253 non-null   object 
 3   Job Description    2253 non-null   object 
 4   Rating             2253 non-null   float64
 5   Company Name       2252 non-null   object 
 6   Location           2253 non-null   object 
 7   Headquarters       2253 non-null   object 
 8   Size               2253 non-null   object 
 9   Founded            2253 non-null   int64  
 10  Type of ownership  2253 non-null   object 
 11  Industry           2253 non-null   object 
 12  Sector             2253 non-null   object 
 13  Revenue            2253 non-null   object 
 14  Competitors        2253 non-null   object 
 15  Easy Apply         2253 non-null   object 
dtypes: float64(1), int64(2),

In [4]:
df.isnull().sum()

Unnamed: 0           0
Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         1
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
Easy Apply           0
dtype: int64

In [9]:
# Step 1: Clean basic characters from the Salary column
df['salary_clean'] = df['Salary Estimate'].str.split('(').str[0]
df['salary_clean'] = df['salary_clean'].str.replace('$', '', regex=False).str.replace('K', '', regex=False).str.strip()

# Step 2: Split into min and max columns as text first
df['min_salary_text'] = df['salary_clean'].str.split('-').str[0]
df['max_salary_text'] = df['salary_clean'].str.split('-').str[1]

# Step 3: Safely convert to float, converting errors or empty strings to NaN automatically
df['min_salary'] = pd.to_numeric(df['min_salary_text'], errors='coerce')
df['max_salary'] = pd.to_numeric(df['max_salary_text'], errors='coerce')

# Step 4: Drop any rows where salary conversion failed (NaN values)
df = df.dropna(subset=['min_salary', 'max_salary'])

# Step 5: Calculate the average salary in thousands
df['avg_salary'] = ((df['min_salary'] + df['max_salary']) / 2) * 1000

# Step 6: Verify the results
df[['Salary Estimate', 'min_salary', 'max_salary', 'avg_salary']].head()

,Salary Estimate,min_salary,max_salary,avg_salary
0,$37K-$66K (Glassdoor est.),37.0,66,51500.0
1,$37K-$66K (Glassdoor est.),37.0,66,51500.0
2,$37K-$66K (Glassdoor est.),37.0,66,51500.0
3,$37K-$66K (Glassdoor est.),37.0,66,51500.0
4,$37K-$66K (Glassdoor est.),37.0,66,51500.0


In [10]:
# Step 1: Convert the job description text to lowercase to ensure uniform matching
df['Job Description'] = df['Job Description'].str.lower()

# Step 2: Search for key skills and create boolean/integer flags (1 for True, 0 for False)
df['python_req'] = df['Job Description'].str.contains('python', na=False).astype(int)
df['sql_req'] = df['Job Description'].str.contains('sql', na=False).astype(int)
df['powerbi_req'] = df['Job Description'].str.contains('power bi|powerbi', na=False).astype(int)

# Step 3: Display a sample to verify the skills extraction
df[['Job Title', 'python_req', 'sql_req', 'powerbi_req']].head()

,Job Title,python_req,sql_req,powerbi_req
0,"Data Analyst, Center on Immigration and Justic...",1,1,0
1,Quality Data Analyst,0,1,0
2,"Senior Data Analyst, Insights & Analytics Team...",1,1,0
3,Data Analyst,0,1,1
4,Reporting Data Analyst,1,1,0


In [11]:
# Save the cleaned dataframe to a new CSV file for Power BI
df.to_csv('Cleaned_Job_Market_Data.csv', index=False)